In [1]:
import cobra
import escher
import pandas as pd
import numpy as np
import json
import csv
from cobra.io import save_json_model
from cobra import Reaction
from cobra.flux_analysis.loopless import add_loopless, loopless_solution
from scipy.stats import linregress

In [2]:
cobra.Configuration().solver = "cplex"
#cobra.Configuration().solver = "gurobi"

In [3]:
#model_path = './models/iMT1026v3.xml'
#model_path = 'iPichia.xml'
model_path = './models/iMT1026v3jup.xml'

model = cobra.io.read_sbml_model(model_path)

model

Name,iMT1026v3
Memory address,23f4e5c6ac8
Number of metabolites,1706
Number of reactions,2237
Number of genes,1026
Number of groups,77
Objective expression,1.0*Ex_biomass - 1.0*Ex_biomass_reverse_5354f
Compartments,"Vacuole, Cytosol, Mitochondria, Peroxisome, Extracellular space, Endoplasmic Reticulum, Golgi Apparatus, Nucleus, Mitochondrial intermembrane space"


In [4]:
#model.solver.problem.Params.NumericFocus = 2
#model.solver.problem.Params.Method = 1
#model.solver.problem.Params.DualReductions = 1
#model.solver.problem.Params.Aggregate = 1

In [5]:
# Change grwoth on glycerol to growth on 60% glucose and 40% methanol
# Add reaction describing this growth

biomass_gly = model.reactions.get_by_id('BIOMASS_glyc')
biomass_gly.bounds = (0,0)
biomass_gly

carbs = model.metabolites.get_by_id('CARBOHYDRATES_c')
DNA = model.metabolites.get_by_id('DNA_c')
lipids = model.metabolites.get_by_id('LIPIDS_c')
prot = model.metabolites.get_by_id('PROTEIN_c')
RNA = model.metabolites.get_by_id('RNA_c')
atp = model.metabolites.get_by_id('atp_c')
cof = model.metabolites.get_by_id('cof_c')
h2o = model.metabolites.get_by_id('h2o_c')
adp = model.metabolites.get_by_id('adp_c')
biomass = model.metabolites.get_by_id('biomass_c')
h = model.metabolites.get_by_id('h_c')
pi = model.metabolites.get_by_id('pi_c')

biomass_glc_meoh = Reaction ('biomass_glc_meoh_60/40')
biomass_glc_meoh.name = 'Biomass composition (g/g) - 60/40 Glucose/Methanol'
biomass_glc_meoh.add_metabolites({carbs: -0.33,
                                   DNA: -0.001,
                                   lipids: -0.042,
                                   prot: -0.49,
                                   RNA: -0.058,
                                   atp: -63.85,
                                   cof: -1,
                                   h2o: -63.85,
                                   adp: 63.85,
                                   pi: 63.85,
                                   biomass: 1,
                                   h: 63.85})
biomass_glc_meoh

Reaction identifier,biomass_glc_meoh_60/40
Name,Biomass composition (g/g) - 60/40 Glucose/Methanol
Memory address,0x23f56d11a88
Stoichiometry,0.33 CARBOHYDRATES_c + 0.001 DNA_c + 0.042 LIPIDS_c + 0.49 PROTEIN_c + 0.058 RNA_c + 63.85 atp_c + cof_c + 63.85 h2o_c --> 63.85 adp_c + biomass_c + 63.85 h_c + 63.85 pi_c 0.33 Carbohydrates + 0.001 DNA + 0.042 Lipids + 0.49 PROTEIN + 0.058 RNA + 63.85 ATP + Cofactors and small molecules + 63.85 H2O --> 63.85 ADP + Biomass + 63.85 H+ + 63.85 Phosphate
GPR,
Lower bound,0.0
Upper bound,1000.0


In [6]:
# Add new biomass reaction to the model

print (len(model.reactions))
model.add_reactions([biomass_glc_meoh])
print (len(model.reactions))

2237
2238


In [7]:
#Tanquem l'entrada de Glicerol
glycerol_exchange = model.exchanges.get_by_id('Ex_glyc')
glycerol_exchange.bounds = (0,0)
glycerol_exchange

Reaction identifier,Ex_glyc
Name,Glycerol exchange
Memory address,0x23f577c5f48
Stoichiometry,glyc_e --> Glycerol -->
GPR,
Lower bound,0
Upper bound,0


In [8]:
methanol_exchange = model.exchanges.get_by_id('Ex_meoh')
methanol_exchange.bounds = (-1.24, -1.18) #Glucose60_Methanol 40
methanol_exchange

Reaction identifier,Ex_meoh
Name,Methanol exchange
Memory address,0x23f577ce808
Stoichiometry,meoh_e <-- Methanol <--
GPR,
Lower bound,-1.24
Upper bound,-1.18


In [9]:
glucose_exchange = model.reactions.get_by_id('Ex_glc_D')
glucose_exchange.bounds = (-0.74, -0.72) #Glucose20_Methanol80
glucose_exchange

Reaction identifier,Ex_glc_D
Name,D-Glucose exchange
Memory address,0x23f577a9f88
Stoichiometry,glc_D_e <-- D-Glucose <--
GPR,
Lower bound,-0.74
Upper bound,-0.72


In [10]:
# Change the reactions for the synthesis of lipids, proteins and sterols from glycerol to those from glucose

model.reactions.get_by_id('LIPIDS_glyc').bounds = (0,0)
model.reactions.get_by_id('PROTEINS_glyc').bounds = (0,0)
model.reactions.get_by_id('STEROLS_glyc').bounds = (0,0)

# note: these reactions from glucose do not have the glucose specification
model.reactions.get_by_id('LIPIDS').bounds = (0,1000)
model.reactions.get_by_id('PROTEINS').bounds = (0,1000)
model.reactions.get_by_id('STEROLS').bounds = (0,1000)

In [11]:
# ATP maintenance requirement (NGAME = Non-Growth Associated Maintenance Energy) based on previous studies on 
# the growth of X33-ROL on Gluc/MeOH from the group (Eric's Master Thesis)

model.reactions.get_by_id('ATPM').bounds = (1.96, 1000)

In [12]:
rolAA = model.reactions.get_by_id('rolAA')
rolAA.bounds = (0,1000)
rolRNA = model.reactions.get_by_id('rolRNA')
rolRNA.bounds = (0,1000)
rolDNA = model.reactions.get_by_id('rolDNA')
rolDNA.bounds = (0,1000)
pROL = model.reactions.get_by_id('pROL')
pROL.bounds = (0,1000)
Rol_transport =  model.reactions.get_by_id('ROLt')
Rol_transport.bounds = (0,1000)
ROL_exchange = model.exchanges.get_by_id('Ex_rol')
#ROL_exchange.bounds = (0.0003773,1000)
ROL_exchange.bounds = (0.003,1000) # Inferit de Bradford

pFAB = model.reactions.get_by_id('pFAB')
pFAB.bounds = (0,0)
fabAA = model.reactions.get_by_id('fabAA')
fabAA.bounds = (0,0)
fabt = model.reactions.get_by_id('fabt')
fabt.bounds = (0,0)
fabRNA = model.reactions.get_by_id('fabRNA')
fabRNA.bounds = (0,0)
fabDNA = model.reactions.get_by_id('fabDNA')
FAB_exchange = model.exchanges.get_by_id('Ex_fab')
FAB_exchange.bounds = (0,0)


# Extra reactions that must be closed for simulations to run smoothly:

APAT2r = model.reactions.get_by_id('APAT2r')
APAT2r.bounds = (0,0) # reaction not present in Pichia, it is yet to be removed from the model

MMSAD3 = model.reactions.get_by_id('MMSAD3')
MMSAD3.bounds = (0,0) # The reduction reaction of MSA into Acetil-CoA it is due to an unspecific effect. Reaction
# under evaluation of being kept or not.

In [13]:
#Creating the cPOS5 reaction
atp_c = model.metabolites.get_by_id('atp_c')
nadh_c = model.metabolites.get_by_id('nadh_c')
adp_c = model.metabolites.get_by_id('adp_c')
nadph_c = model.metabolites.get_by_id('nadph_c')
h_c = model.metabolites.get_by_id('h_c')

cPOS5 = Reaction('cPOS5')
cPOS5.name = 'Cytosolic NADH Kinase from Saccharomyces cerevisiae'
cPOS5.add_metabolites({atp_c: -1, 
                           nadh_c: -1, 
                           adp_c: 1, 
                           nadph_c: 1,
                               h_c: 1})
cPOS5.bounds = (0,0)
print (cPOS5.reaction)
cPOS5

# from Saccharomyces. Found in Brenda ('for references in articles please use BRENDA:EC2.7.1.86')
# Also in Uniprot with ID Q06892 and ID YPL188W (Saccharomyces Genome Database)
# It is a kinase from the mitochondrial matrix (Outten and Culotta, 2003)
# Although it is a mitochondrial enzyme in Saccharomyces, Marius et al. (2) added a tag to express it in the cytosol
# you sure about the above? It says in page of his thesis that it was truncated: they removed the first 49 bp of the
# gene in order to remove the peptide leading sequence, ensuring that it would stay in the cytosol and not go to the
# mitochondria

atp_c + nadh_c --> adp_c + h_c + nadph_c


Reaction identifier,cPOS5
Name,Cytosolic NADH Kinase from Saccharomyces cerevisiae
Memory address,0x23f090a5a48
Stoichiometry,atp_c + nadh_c --> adp_c + h_c + nadph_c ATP + NADH --> ADP + H+ + NADPH
GPR,
Lower bound,0
Upper bound,0


In [14]:
print (len(model.reactions))
model.add_reactions([cPOS5])
print (len(model.reactions))

2238
2239


In [15]:
# Creating the cPntAB reaction

nad_c = model.metabolites.get_by_id('nad_c')
nadh_c = model.metabolites.get_by_id('nadh_c')
nadp_c = model.metabolites.get_by_id('nadp_c')
nadph_c = model.metabolites.get_by_id('nadph_c')

cPntAB = Reaction('cPntAB')
cPntAB.name = 'Cytosolic Nicotinamide Nucleotide Transhydrogenase'

cPntAB.add_metabolites({
    nadh_c: -1,
    nadp_c: -1,
    nad_c: 1,
    nadph_c: 1
})

cPntAB.bounds = (0, 0)

print(cPntAB.reaction)
cPntAB

nadh_c + nadp_c --> nad_c + nadph_c


Reaction identifier,cPntAB
Name,Cytosolic Nicotinamide Nucleotide Transhydrogenase
Memory address,0x23f577994c8
Stoichiometry,nadh_c + nadp_c --> nad_c + nadph_c NADH + NADP --> NAD+ + NADPH
GPR,
Lower bound,0
Upper bound,0


In [16]:
print (len(model.reactions))
model.add_reactions([cPntAB])
print (len(model.reactions))

2239
2240


In [17]:
# Creating the cNoxE reaction

nad_c = model.metabolites.get_by_id('nad_c')
nadh_c = model.metabolites.get_by_id('nadh_c')
o2_c = model.metabolites.get_by_id('o2_c')
h2o_c = model.metabolites.get_by_id('h2o_c')
h_c = model.metabolites.get_by_id('h_c')

cNoxE = Reaction('cNoxE')
cNoxE.name = 'Water-forming NADH Oxidase from Lactococcus lactis'

cNoxE.add_metabolites({
    nadh_c: -1,
    h_c: -1,
    o2_c: -0.5,
    nad_c: 1,
    h2o_c: 1
})

cNoxE.bounds = (0, 0)

print(cNoxE.reaction)
cNoxE

h_c + nadh_c + 0.5 o2_c --> h2o_c + nad_c


Reaction identifier,cNoxE
Name,Water-forming NADH Oxidase from Lactococcus lactis
Memory address,0x23f5791c048
Stoichiometry,h_c + nadh_c + 0.5 o2_c --> h2o_c + nad_c H+ + NADH + 0.5 Oxygen --> H2O + NAD+
GPR,
Lower bound,0
Upper bound,0


In [18]:
print (len(model.reactions))
model.add_reactions([cNoxE])
print (len(model.reactions))

2240
2241


In [19]:
# REACTION RATIOS PER GLUCOSA METANOL

ReactionRatio1 = model.problem.Constraint(model.reactions.CSm.flux_expression - model.reactions.ACONTm.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio1)

ReactionRatio2 = model.problem.Constraint(model.reactions.AKGDam.flux_expression - model.reactions.FUMm.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio2)

ReactionRatio8 = model.problem.Constraint(68*model.reactions.AKGDam.flux_expression - 46*model.reactions.CSm.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio8)

ReactionRatio9 = model.problem.Constraint(model.reactions.G6PDH2.flux_expression - model.reactions.GND.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio9)

ReactionRatio10 = model.problem.Constraint(0.8*model.reactions.MDHm.flux_expression - model.reactions.FUMm.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio10)

ReactionRatio12 = model.problem.Constraint(35*model.reactions.PYK.flux_expression - 143*model.reactions.PC.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio12)

ReactionRatio13 = model.problem.Constraint(68*model.reactions.PYK.flux_expression - 143*model.reactions.PYRt2m.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio13)


ReactionRatio14 = model.problem.Constraint(40*model.reactions.GAPD.flux_expression - 145*model.reactions.FBA.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio14)


ReactionRatio16 = model.problem.Constraint(model.reactions.GAPD.flux_expression - model.reactions.PYK.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio16)

ReactionRatio20 = model.problem.Constraint(0.18*model.reactions.FALDtx.flux_expression - 0.82*model.reactions.DAS.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio20)

ReactionRatio21 = model.problem.Constraint(model.reactions.G6PDH2.flux_expression + 0.4*model.reactions.Ex_glc_D.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio21)

In [20]:
pfba_WT = cobra.flux_analysis.pfba(model)
#pfba_WT = loopless_solution(model)

In [21]:
pfba_WT.fluxes['Ex_rol']

0.003

In [22]:
pfba_WT.fluxes['Ex_biomass']

0.0825492805746326

In [23]:
# Remove Reaction Ratio Constraints before starting MOMA simulations

ReactionRatioList = [ReactionRatio1, ReactionRatio2, ReactionRatio8, ReactionRatio9, ReactionRatio10,
                     ReactionRatio12, ReactionRatio13, ReactionRatio14, ReactionRatio16, ReactionRatio20, ReactionRatio21]
                    

model.remove_cons_vars(ReactionRatioList)

In [24]:
NewRolProd = (0.0033, 0.0036, 0.0039, 0.0042, 0.0045, 0.0048)

In [25]:
#MOMA non-linear Simulations 
MomaResults = []
for x in NewRolProd:
        ROL_exchange.bounds = (x, x)
        
        # Perform MOMA
        moma_result = cobra.flux_analysis.moma(model,pfba_WT,0) #Boolean per si el MOMA es lineal o no
        MomaResults.append(moma_result.fluxes)

In [26]:
decreased_genes = []
gene_slopes = {}
gene_r_squared = {}

# Thresholds
flux_tolerance = 1e-6
slope_threshold = -0.05
r2_threshold = 0.80

# Collect genes present in the model
model_genes = list(model.genes)

for gene in model_genes:

    gene_id = gene.id

    # Reactions associated with this gene
    associated_reactions = list(gene.reactions)

    # Keep only genes associated with exactly one reaction
    if len(associated_reactions) != 1:
        continue

    # Calculate gene-associated flux for each MOMA solution
    reaction = associated_reactions[0]

    gene_fluxes = np.array([
        abs(solution[reaction.id])
        for solution in MomaResults
    ], dtype=float)

    # Discard genes with negligible flux across all solutions
    if np.max(np.abs(gene_fluxes)) < flux_tolerance:
        continue

    # Fit gene-associated flux against ROL production
    slope, intercept, r_value, p_value, std_err = linregress(
        NewRolProd,
        gene_fluxes
    )

    # Store slope and R²
    gene_slopes[gene_id] = slope
    gene_r_squared[gene_id] = r_value**2

    # Identify genes whose associated flux decreases
    if (
        slope < slope_threshold
        and r_value**2 > r2_threshold
    ):
        decreased_genes.append(gene_id)

print(f"Number of decreased genes: {len(decreased_genes)}")
print(f"Slope threshold: {slope_threshold}")
print(f"R² threshold: {r2_threshold}")

print("\nDecreased genes:")
for gene_id in decreased_genes:
    print(
        f"{gene_id}: "
        f"slope = {gene_slopes[gene_id]:.6g}, "
        f"R² = {gene_r_squared[gene_id]:.4f}"
    )

Number of decreased genes: 145
Slope threshold: -0.05
R² threshold: 0.8

Decreased genes:
PAS_chr4_0535: slope = -0.236755, R² = 1.0000
PAS_chr3_1199: slope = -0.236755, R² = 1.0000
PAS_chr2-2_0462: slope = -0.236755, R² = 1.0000
PAS_chr4_0355: slope = -0.236755, R² = 1.0000
PAS_chr4_0142: slope = -0.236755, R² = 1.0000
PAS_chr1-3_0210: slope = -0.236755, R² = 1.0000
PAS_chr3_0907: slope = -0.236755, R² = 1.0000
PAS_chr2-2_0169: slope = -0.236755, R² = 1.0000
PAS_chr1-4_0371: slope = -0.236755, R² = 1.0000
PAS_chr4_0283: slope = -0.236755, R² = 1.0000
PAS_chr3_0104: slope = -0.236755, R² = 1.0000
PAS_chr4_0960: slope = -0.236755, R² = 1.0000
PAS_chr1-4_0274: slope = -0.236755, R² = 1.0000
PAS_chr3_1188: slope = -0.236755, R² = 1.0000
PAS_chr1-1_0172: slope = -0.236755, R² = 1.0000
PAS_chr2-1_0456: slope = -0.236755, R² = 1.0000
PAS_chr1-1_0124: slope = -0.236755, R² = 1.0000
PAS_chr1-4_0039: slope = -0.236755, R² = 1.0000
PAS_chr4_0245: slope = -0.236755, R² = 1.0000
PAS_chr3_0911: slo

In [27]:
ExoFlux = 0.1
GrowthLimit = 0.02

In [28]:
essential_genes = []
fba_knockdown_growth_results = {}

ROL_exchange.bounds = (0, 1000)

for gene_id in decreased_genes:

    print(f"Testing gene {gene_id}...")

    with model:

        gene = model.genes.get_by_id(gene_id)

        # Knock out the gene using its GPR rules
        gene.knock_out()

        # Run FBA
        solution = model.optimize()

        # Check solver status
        status = solution.status

        print(f"    Solver status: {status}")

        if status != "optimal":
            print(
                f"    {gene_id} → {status.upper()} → ESSENTIAL"
            )

            essential_genes.append(gene_id)
            fba_knockdown_growth_results[gene_id] = status
            continue

        # Retrieve growth only for an optimal solution
        growth = solution.fluxes["Ex_biomass"]

        print(f"    Growth = {growth}")

        fba_knockdown_growth_results[gene_id] = growth

        if growth < GrowthLimit:
            print(f"    {gene_id} → ESSENTIAL")
            essential_genes.append(gene_id)

print("\n========================================")
print(f"Genes tested: {len(decreased_genes)}")
print(f"Essential genes: {len(essential_genes)}")
print("========================================")

print("\nEssential genes:")
for gene_id in essential_genes:
    print(gene_id)

Testing gene PAS_chr4_0535...
    Solver status: optimal
    Growth = 0.08331556127385412
Testing gene PAS_chr3_1199...
    Solver status: optimal
    Growth = 0.08331556127385412
Testing gene PAS_chr2-2_0462...
    Solver status: optimal
    Growth = 0.08331556127385412
Testing gene PAS_chr4_0355...
    Solver status: optimal
    Growth = 0.08331556127385412
Testing gene PAS_chr4_0142...
    Solver status: optimal
    Growth = 0.08331556127385412
Testing gene PAS_chr1-3_0210...
    Solver status: optimal
    Growth = 0.08331556127385412
Testing gene PAS_chr3_0907...
    Solver status: optimal
    Growth = 0.08331556127385412
Testing gene PAS_chr2-2_0169...
    Solver status: optimal
    Growth = 0.08331556127385412
Testing gene PAS_chr1-4_0371...
    Solver status: optimal
    Growth = 0.08331556127385412
Testing gene PAS_chr4_0283...
    Solver status: optimal
    Growth = 0.08331556127385412
Testing gene PAS_chr3_0104...
    Solver status: optimal
    Growth = 0.08331556127385412
Te

C:\Users\Eric\miniconda3\envs\s2m2\lib\site-packages\cobra\util\solver.py:554: UserWarning: Solver status is 'infeasible'.
  warn(f"Solver status is '{status}'.", UserWarning)


    Solver status: infeasible
    PAS_chr1-4_0615 → INFEASIBLE → ESSENTIAL
Testing gene PAS_chr2-1_0313...
    Solver status: optimal
    Growth = 0.09803021834113752
Testing gene PAS_chr2-1_0199...
    Solver status: optimal
    Growth = 0.09880219661332013
Testing gene PAS_chr4_0259...
    Solver status: optimal
    Growth = 0.09880219661332013
Testing gene PAS_chr3_0574...
    Solver status: optimal
    Growth = 0.09880219661332013
Testing gene PAS_chr2-2_0104...
    Solver status: optimal
    Growth = 0.09880219661332013
Testing gene PAS_chr4_0239...
    Solver status: optimal
    Growth = 0.09880219661332013
Testing gene PAS_chr3_0568...
    Solver status: optimal
    Growth = 0.09880219661332013
Testing gene PAS_chr2-2_0434...
    Solver status: optimal
    Growth = 0.09880219661332013
Testing gene PAS_chr4_0906...
    Solver status: optimal
    Growth = 0.09880219661332013
Testing gene PAS_chr4_0735...
    Solver status: optimal
    Growth = 0.09880219661332013
Testing gene PAS_

    Solver status: optimal
    Growth = 0.09880219615449126
Testing gene PAS_chr2-2_0053...
    Solver status: optimal
    Growth = -0.0
    PAS_chr2-2_0053 → ESSENTIAL
Testing gene PAS_chr2-2_0318...
    Solver status: infeasible
    PAS_chr2-2_0318 → INFEASIBLE → ESSENTIAL
Testing gene PAS_chr1-4_0450...
    Solver status: infeasible
    PAS_chr1-4_0450 → INFEASIBLE → ESSENTIAL
Testing gene PAS_chr1-4_0551...
    Solver status: optimal
    Growth = -0.0
    PAS_chr1-4_0551 → ESSENTIAL
Testing gene PAS_chr1-1_0394...
    Solver status: optimal
    Growth = 0.09880219520452906
Testing gene RPPA11109...
    Solver status: optimal
    Growth = 0.09880219520452906
Testing gene RPPA11110...
    Solver status: optimal
    Growth = 0.09880219520452906
Testing gene PAS_chr3_0867...
    Solver status: optimal
    Growth = 0.09803021698003506
Testing gene PAS_chr2-2_0337...
    Solver status: optimal
    Growth = 0.09880219520446669
Testing gene PAS_chr3_0676...
    Solver status: optimal
    G

In [29]:
# ============================================================
# Test rescue of essential genes by POS5 expression
# ============================================================

rescued_genes_byPOS5 = []
rescue_results_genes = {}

# Express POS5
cPOS5.bounds = (ExoFlux, ExoFlux)

for gene_id in essential_genes:

    print(f"Testing POS5 rescue of gene {gene_id}...")

    with model:

        # Knock out the gene
        gene = model.genes.get_by_id(gene_id)
        gene.knock_out()

        # Run FBA with POS5 expression
        solution = model.optimize()

        status = solution.status

        if status != "optimal":

            rescue_results_genes[gene_id] = {
                "KO_growth": fba_knockdown_growth_results[gene_id],
                "POS5_growth": status,
                "rescued": False
            }

            print(f"    Solver status: {status} → Not rescued")
            continue

        # Retrieve growth
        growth = solution.fluxes["Ex_biomass"]

        # Store results
        rescued = growth >= GrowthLimit

        rescue_results_genes[gene_id] = {
            "KO_growth": fba_knockdown_growth_results[gene_id],
            "POS5_growth": growth,
            "rescued": rescued
        }

        # Add to rescued genes
        if rescued:
            rescued_genes_byPOS5.append(gene_id)
            print(f"    Growth = {growth:.6g} → RESCUED")
        else:
            print(f"    Growth = {growth:.6g} → Not rescued")

# ============================================================
# Summary
# ============================================================

print("\n========================================")
print(f"Essential genes tested: {len(essential_genes)}")
print(f"Genes rescued by POS5: {len(rescued_genes_byPOS5)}")
print("========================================")

print("\nRescued genes:")
for gene_id in rescued_genes_byPOS5:
    print(gene_id)

Testing POS5 rescue of gene PAS_chr4_0210...
    Solver status: infeasible → Not rescued
Testing POS5 rescue of gene PAS_chr1-4_0615...
    Solver status: infeasible → Not rescued
Testing POS5 rescue of gene PAS_chr4_0060...
    Growth = -0 → Not rescued
Testing POS5 rescue of gene PAS_chr1-1_0067...
    Growth = -0 → Not rescued
Testing POS5 rescue of gene PAS_chr3_0085...
    Solver status: infeasible → Not rescued
Testing POS5 rescue of gene PAS_chr2-2_0119...
    Solver status: infeasible → Not rescued
Testing POS5 rescue of gene PAS_chr3_0460...
    Growth = 0.0158173 → Not rescued
Testing POS5 rescue of gene PAS_chr1-1_0484...
    Growth = 0.0158173 → Not rescued
Testing POS5 rescue of gene PAS_chr2-1_0612...
    Growth = 0.0158173 → Not rescued
Testing POS5 rescue of gene PAS_chr2-1_0751...
    Growth = 0.0158173 → Not rescued
Testing POS5 rescue of gene PAS_chr3_0059...
    Growth = 0.0158173 → Not rescued
Testing POS5 rescue of gene PAS_chr4_0892...
    Growth = 0.0158173 → No

In [30]:
moma_rescued_genes_results = {}

# Express POS5 and allow ROL production
cPOS5.bounds = (ExoFlux, ExoFlux)
ROL_exchange.bounds = (0, 1000)

for gene_id in rescued_genes_byPOS5:

    print(f"Testing gene {gene_id}...")

    with model:

        # Knock out the rescued gene
        gene = model.genes.get_by_id(gene_id)
        gene.knock_out()

        # Run MOMA using WT pFBA as reference
        moma_result = cobra.flux_analysis.moma(
            model,
            solution=pfba_WT,
            linear=False
        )

        status = moma_result.status

        if status != "optimal":
            moma_rescued_genes_results[gene_id] = {
                "status": status,
                "growth": np.nan,
                "ROL_production": np.nan
            }

            print(f"{gene_id}: {status.upper()}")
            continue

        # Get biomass and ROL production
        growth = moma_result.fluxes["Ex_biomass"]
        rol_production = moma_result.fluxes["Ex_rol"]

        # Store results
        moma_rescued_genes_results[gene_id] = {
            "status": status,
            "growth": growth,
            "ROL_production": rol_production
        }

        print(
            f"{gene_id}: "
            f"Growth = {growth:.6f}, "
            f"ROL production = {rol_production:.6f}"
        )

Testing gene PAS_chr3_0781...
PAS_chr3_0781: Growth = 0.080142, ROL production = 0.004178
Testing gene PAS_chr4_0847...
PAS_chr4_0847: Growth = 0.080142, ROL production = 0.004178
Testing gene PAS_chr2-2_0266...
PAS_chr2-2_0266: Growth = 0.000000, ROL production = 0.036043


In [31]:
# ============================================================
# Test rescue of essential genes by cPntAB expression
# ============================================================

rescued_genes_bycPntAB = []
rescue_results_genes = {}
cPOS5.bounds = (0, 0)

# Express POS5
cPntAB.bounds = (ExoFlux, ExoFlux)

for gene_id in essential_genes:

    print(f"Testing cPntAB rescue of gene {gene_id}...")

    with model:

        # Knock out the gene
        gene = model.genes.get_by_id(gene_id)
        gene.knock_out()

        # Run FBA with POS5 expression
        solution = model.optimize()

        status = solution.status

        if status != "optimal":

            rescue_results_genes[gene_id] = {
                "KO_growth": fba_knockdown_growth_results[gene_id],
                "cPntAB_growth": status,
                "rescued": False
            }

            print(f"    Solver status: {status} → Not rescued")
            continue

        # Retrieve growth
        growth = solution.fluxes["Ex_biomass"]

        # Store results
        rescued = growth >= GrowthLimit

        rescue_results_genes[gene_id] = {
            "KO_growth": fba_knockdown_growth_results[gene_id],
            "cPntAB_growth": growth,
            "rescued": rescued
        }

        # Add to rescued genes
        if rescued:
            rescued_genes_bycPntAB.append(gene_id)
            print(f"    Growth = {growth:.6g} → RESCUED")
        else:
            print(f"    Growth = {growth:.6g} → Not rescued")

# ============================================================
# Summary
# ============================================================

print("\n========================================")
print(f"Essential genes tested: {len(essential_genes)}")
print(f"Genes rescued by POS5: {len(rescued_genes_bycPntAB)}")
print("========================================")

print("\nRescued genes:")
for gene_id in rescued_genes_bycPntAB:
    print(gene_id)

Testing cPntAB rescue of gene PAS_chr4_0210...
    Solver status: infeasible → Not rescued
Testing cPntAB rescue of gene PAS_chr1-4_0615...
    Solver status: infeasible → Not rescued
Testing cPntAB rescue of gene PAS_chr4_0060...
    Growth = -0 → Not rescued
Testing cPntAB rescue of gene PAS_chr1-1_0067...
    Growth = -0 → Not rescued
Testing cPntAB rescue of gene PAS_chr3_0085...
    Solver status: infeasible → Not rescued
Testing cPntAB rescue of gene PAS_chr2-2_0119...
    Growth = 0.0167533 → Not rescued
Testing cPntAB rescue of gene PAS_chr3_0460...
    Growth = 0.0167533 → Not rescued
Testing cPntAB rescue of gene PAS_chr1-1_0484...
    Growth = 0.0167533 → Not rescued
Testing cPntAB rescue of gene PAS_chr2-1_0612...
    Growth = 0.0167533 → Not rescued
Testing cPntAB rescue of gene PAS_chr2-1_0751...
    Growth = 0.0167533 → Not rescued
Testing cPntAB rescue of gene PAS_chr3_0059...
    Growth = 0.0167533 → Not rescued
Testing cPntAB rescue of gene PAS_chr4_0892...
    Growth

In [32]:
moma_rescued_genes_results = {}

# Express POS5 and allow ROL production
cPntAB.bounds = (ExoFlux, ExoFlux)
ROL_exchange.bounds = (0, 1000)

for gene_id in rescued_genes_bycPntAB:

    print(f"Testing gene {gene_id}...")

    with model:

        # Knock out the rescued gene
        gene = model.genes.get_by_id(gene_id)
        gene.knock_out()

        # Run MOMA using WT pFBA as reference
        moma_result = cobra.flux_analysis.moma(
            model,
            solution=pfba_WT,
            linear=False
        )

        status = moma_result.status

        if status != "optimal":
            moma_rescued_genes_results[gene_id] = {
                "status": status,
                "growth": np.nan,
                "ROL_production": np.nan
            }

            print(f"{gene_id}: {status.upper()}")
            continue

        # Get biomass and ROL production
        growth = moma_result.fluxes["Ex_biomass"]
        rol_production = moma_result.fluxes["Ex_rol"]

        # Store results
        moma_rescued_genes_results[gene_id] = {
            "status": status,
            "growth": growth,
            "ROL_production": rol_production
        }

        print(
            f"{gene_id}: "
            f"Growth = {growth:.6f}, "
            f"ROL production = {rol_production:.6f}"
        )

Testing gene PAS_chr4_0847...
PAS_chr4_0847: Growth = 0.082186, ROL production = 0.003139
Testing gene PAS_chr3_0100...
PAS_chr3_0100: Growth = 0.082186, ROL production = 0.003139
Testing gene PAS_chr2-2_0266...
PAS_chr2-2_0266: Growth = 0.000000, ROL production = 0.036447


In [33]:
# ============================================================
# Test rescue of essential genes by cNoxE expression
# ============================================================

rescued_genes_bycNoxE = []
rescue_results_genes = {}
cPntAB.bounds = (0, 0)

# Express POS5
cNoxE.bounds = (ExoFlux, ExoFlux)

for gene_id in essential_genes:

    print(f"Testing cPntAB rescue of gene {gene_id}...")

    with model:

        # Knock out the gene
        gene = model.genes.get_by_id(gene_id)
        gene.knock_out()

        # Run FBA with POS5 expression
        solution = model.optimize()

        status = solution.status

        if status != "optimal":

            rescue_results_genes[gene_id] = {
                "KO_growth": fba_knockdown_growth_results[gene_id],
                "cPntAB_growth": status,
                "rescued": False
            }

            print(f"    Solver status: {status} → Not rescued")
            continue

        # Retrieve growth
        growth = solution.fluxes["Ex_biomass"]

        # Store results
        rescued = growth >= GrowthLimit

        rescue_results_genes[gene_id] = {
            "KO_growth": fba_knockdown_growth_results[gene_id],
            "cNoxE_growth": growth,
            "rescued": rescued
        }

        # Add to rescued genes
        if rescued:
            rescued_genes_bycNoxE.append(gene_id)
            print(f"    Growth = {growth:.6g} → RESCUED")
        else:
            print(f"    Growth = {growth:.6g} → Not rescued")

# ============================================================
# Summary
# ============================================================

print("\n========================================")
print(f"Essential genes tested: {len(essential_genes)}")
print(f"Genes rescued by POS5: {len(rescued_genes_bycNoxE)}")
print("========================================")

print("\nRescued genes:")
for gene_id in rescued_genes_bycNoxE:
    print(gene_id)

Testing cPntAB rescue of gene PAS_chr4_0210...
    Solver status: infeasible → Not rescued
Testing cPntAB rescue of gene PAS_chr1-4_0615...
    Solver status: infeasible → Not rescued
Testing cPntAB rescue of gene PAS_chr4_0060...
    Growth = -0 → Not rescued
Testing cPntAB rescue of gene PAS_chr1-1_0067...
    Growth = -0 → Not rescued
Testing cPntAB rescue of gene PAS_chr3_0085...
    Solver status: infeasible → Not rescued
Testing cPntAB rescue of gene PAS_chr2-2_0119...
    Growth = 0.0162853 → Not rescued
Testing cPntAB rescue of gene PAS_chr3_0460...
    Growth = 0.0162853 → Not rescued
Testing cPntAB rescue of gene PAS_chr1-1_0484...
    Growth = 0.0162853 → Not rescued
Testing cPntAB rescue of gene PAS_chr2-1_0612...
    Growth = 0.0162853 → Not rescued
Testing cPntAB rescue of gene PAS_chr2-1_0751...
    Growth = 0.0162853 → Not rescued
Testing cPntAB rescue of gene PAS_chr3_0059...
    Growth = 0.0162853 → Not rescued
Testing cPntAB rescue of gene PAS_chr4_0892...
    Growth

In [34]:
moma_rescued_genes_results = {}

# Express POS5 and allow ROL production
cNoxE.bounds = (ExoFlux, ExoFlux)
ROL_exchange.bounds = (0, 1000)

for gene_id in rescued_genes_bycNoxE:

    print(f"Testing gene {gene_id}...")

    with model:

        # Knock out the rescued gene
        gene = model.genes.get_by_id(gene_id)
        gene.knock_out()

        # Run MOMA using WT pFBA as reference
        moma_result = cobra.flux_analysis.moma(
            model,
            solution=pfba_WT,
            linear=False
        )

        status = moma_result.status

        if status != "optimal":
            moma_rescued_genes_results[gene_id] = {
                "status": status,
                "growth": np.nan,
                "ROL_production": np.nan
            }

            print(f"{gene_id}: {status.upper()}")
            continue

        # Get biomass and ROL production
        growth = moma_result.fluxes["Ex_biomass"]
        rol_production = moma_result.fluxes["Ex_rol"]

        # Store results
        moma_rescued_genes_results[gene_id] = {
            "status": status,
            "growth": growth,
            "ROL_production": rol_production
        }

        print(
            f"{gene_id}: "
            f"Growth = {growth:.6f}, "
            f"ROL production = {rol_production:.6f}"
        )

Testing gene PAS_chr3_0781...
PAS_chr3_0781: Growth = 0.081368, ROL production = 0.002141
Testing gene PAS_chr4_0847...
PAS_chr4_0847: Growth = 0.081368, ROL production = 0.002141
Testing gene PAS_chr3_0100...
PAS_chr3_0100: Growth = 0.081368, ROL production = 0.002141
Testing gene PAS_chr2-2_0266...
PAS_chr2-2_0266: Growth = 0.000000, ROL production = 0.035928


In [35]:
moma_rescued_genes_results = {}

# Express POS5 and allow ROL production
cNoxE.bounds = (0, 0)
ROL_exchange.bounds = (0, 1000)

for gene_id in rescued_genes_bycNoxE:

    print(f"Testing gene {gene_id}...")

    with model:

        # Knock out the rescued gene
        gene = model.genes.get_by_id(gene_id)
        gene.knock_out()

        # Run MOMA using WT pFBA as reference
        moma_result = cobra.flux_analysis.moma(
            model,
            solution=pfba_WT,
            linear=True
        )

        status = moma_result.status

        if status != "optimal":
            moma_rescued_genes_results[gene_id] = {
                "status": status,
                "growth": np.nan,
                "ROL_production": np.nan
            }

            print(f"{gene_id}: {status.upper()}")
            continue

        # Get biomass and ROL production
        growth = moma_result.fluxes["Ex_biomass"]
        rol_production = moma_result.fluxes["Ex_rol"]

        # Store results
        moma_rescued_genes_results[gene_id] = {
            "status": status,
            "growth": growth,
            "ROL_production": rol_production
        }

        print(
            f"{gene_id}: "
            f"Growth = {growth:.6f}, "
            f"ROL production = {rol_production:.6f}"
        )

Testing gene PAS_chr3_0781...
PAS_chr3_0781: Growth = 0.082549, ROL production = 0.003000
Testing gene PAS_chr4_0847...
PAS_chr4_0847: Growth = 0.082549, ROL production = 0.003000
Testing gene PAS_chr3_0100...
PAS_chr3_0100: Growth = 0.082549, ROL production = 0.003000
Testing gene PAS_chr2-2_0266...
PAS_chr2-2_0266: Growth = 0.000000, ROL production = 0.022937
